In [2]:
!pip install --upgrade pip setuptools wheel


In [3]:
!pip install pyarrow --only-binary=:all:


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 78.3 MB/s  0:00:00 eta 0:00:01


In [5]:
!pip install torch transformers accelerate sentencepiece
!pip install datasets pyarrow==14.0.2 --only-binary=:all:


  Using cached datasets-4.4.2-py3-none-any.whl.metadata (19 kB)
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
  Using cached datasets-4.4.1-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.4.0-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached datasets-4.2.0-py3-none-any.whl.metadata (18 kB)
  Using cached datasets-4.1.1-py3-none-any.whl.metadata (18 kB)
  Using cached datasets-4.1.0-py3-none-any.whl.metadata (18 kB)
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
INFO: pip is still looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidanc

In [6]:
import pyarrow
import datasets

print(pyarrow.__version__)
print(datasets.__version__)


<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.Tensor size changed, may indicate binary incompatibility. Expected 64 from C header, got 80 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.ChunkedArray size changed, may indicate binary incompatibility. Expected 64 from C header, got 72 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib._Tabular size changed, may indicate binary incompatibility. Expected 24 from C header, got 32 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.Table size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.NativeFile size changed, may indicate binary incompatibility. Expected 96 from C header, got 104 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.BufferedInputStream size changed, may indicate binary incompatibility. Expected 

20.0.0
2.19.2


In [3]:
# ============================================================
# FINAL HYDRALORA SCRIPT — NAN-SAFE, STABLE, SINGLE BLOCK
# Mistral-7B | A100 80GB | BF16 | Research-Grade
# ============================================================

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader

device = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True

# =========================
# Tokenizer
# =========================
model_name = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False
)
tokenizer.pad_token = tokenizer.eos_token

# =========================
# Model
# =========================
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_flash_attention_2 = True
model.config.use_cache = False

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# =========================
# HydraLoRA
# =========================
class HydraLoRA(nn.Module):
    def __init__(self, in_features, out_features, r=8, num_experts=8, alpha=32, dropout=0.05):
        super().__init__()
        self.scaling = alpha / r

        self.lora_A = nn.ModuleList([
            nn.Linear(in_features, r, bias=False)
            for _ in range(num_experts)
        ])
        self.lora_B = nn.ModuleList([
            nn.Linear(r, out_features, bias=False)
            for _ in range(num_experts)
        ])

        self.router = nn.Linear(in_features, num_experts)
        self.dropout = nn.Dropout(dropout)

        for A, B in zip(self.lora_A, self.lora_B):
            nn.init.kaiming_uniform_(A.weight, a=5 ** 0.5)
            nn.init.zeros_(B.weight)

        nn.init.zeros_(self.router.weight)
        nn.init.zeros_(self.router.bias)

    def forward(self, x):
        # ---- Stable router ----
        logits = self.router(x)
        logits = logits - logits.max(dim=-1, keepdim=True).values
        logits = torch.clamp(logits, -20, 20)
        gates = F.softmax(logits, dim=-1)

        expert_outs = []
        for A, B in zip(self.lora_A, self.lora_B):
            expert_outs.append(B(self.dropout(A(x))) * self.scaling)

        expert_outs = torch.stack(expert_outs, dim=-2)
        return torch.sum(gates.unsqueeze(-1) * expert_outs, dim=-2)

class HydraLoRALinear(nn.Module):
    def __init__(self, base_layer):
        super().__init__()
        self.base = base_layer

        self.hydra = HydraLoRA(
            base_layer.in_features,
            base_layer.out_features
        )

        # Device + dtype alignment
        self.hydra.to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype
        )

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.base(x) + self.hydra(x)

# =========================
# Inject HydraLoRA into Q,V
# =========================
def apply_hydralora(model):
    for name, module in model.named_modules():
        if name.endswith("q_proj") or name.endswith("v_proj"):
            parent = model
            *path, attr = name.split(".")
            for p in path:
                parent = getattr(parent, p)
            setattr(parent, attr, HydraLoRALinear(getattr(parent, attr)))

apply_hydralora(model)

# =========================
# Freeze non-Hydra params
# =========================
for name, param in model.named_parameters():
    param.requires_grad = ("hydra" in name.lower())

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

# =========================
# Dataset
# =========================
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding=False
    )

tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"]
)
tokenized.set_format(type="torch")

def collate_fn(batch):
    return {
        "input_ids": torch.nn.utils.rnn.pad_sequence(
            [x["input_ids"] for x in batch],
            batch_first=True,
            padding_value=tokenizer.pad_token_id
        )
    }

train_loader = DataLoader(
    tokenized["train"],
    batch_size=1,
    shuffle=True,
    collate_fn=collate_fn
)

# =========================
# Optimizer (stable LR)
# =========================
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5
)

# =========================
# Training (NaN-SAFE)
# =========================
model.train()

GRAD_ACC = 8
MAX_STEPS = 500
step = 0
skipped = 0

for batch in train_loader:
    input_ids = batch["input_ids"].to(device)

    # ---- Skip empty / tiny batches ----
    if input_ids.numel() == 0 or input_ids.shape[1] < 2:
        skipped += 1
        continue

    outputs = model(input_ids=input_ids, labels=input_ids)
    loss = outputs.loss

    # ---- Skip NaN / Inf losses ----
    if torch.isnan(loss) or torch.isinf(loss):
        optimizer.zero_grad(set_to_none=True)
        skipped += 1
        continue

    loss = loss / GRAD_ACC
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        filter(lambda p: p.requires_grad, model.parameters()),
        max_norm=1.0
    )

    if (step + 1) % GRAD_ACC == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    if step % 50 == 0:
        print(f"Step {step} | Loss: {loss.item() * GRAD_ACC:.4f}")

    step += 1
    if step >= MAX_STEPS:
        break

print(f"✅ Training finished | Skipped batches: {skipped}")


Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.40s/it]


Trainable params: 29,360,640
Trainable %: 0.4038%
Step 0 | Loss: 5.5946
Step 50 | Loss: 2.6866
Step 100 | Loss: 2.0988
Step 150 | Loss: 2.2924
Step 200 | Loss: 1.6744
Step 250 | Loss: 2.8313
Step 300 | Loss: 1.6380
Step 350 | Loss: 1.2959
Step 400 | Loss: 1.8854
Step 450 | Loss: 5.6311
✅ Training finished | Skipped batches: 252


In [4]:
# Router entropy (lower over time = specialization)
with torch.no_grad():
    x = torch.randn(2, 16, model.config.hidden_size, device="cuda", dtype=torch.bfloat16)
    logits = model.model.layers[0].self_attn.q_proj.hydra.router(x)
    probs = torch.softmax(logits, dim=-1)
    entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=-1).mean()
    print("Router entropy:", entropy.item())


Router entropy: 2.078125


PAPER-FAITHFUL HYDRALORA CODE

In [1]:
# ============================================================
# PAPER-FAITHFUL HYDRALORA (TOP-2 ROUTING + ENTROPY METRICS)
# Mistral-7B | A100 80GB | BF16 | Research-grade
# ============================================================

import os, math
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader

# -------------------------
# Global setup
# -------------------------
device = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True

model_name = "mistralai/Mistral-7B-v0.1"

# -------------------------
# Tokenizer
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False
)
tokenizer.pad_token = tokenizer.eos_token

# -------------------------
# Model
# -------------------------
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_flash_attention_2 = True
model.config.use_cache = False

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# ============================================================
# HydraLoRA (Top-2 Sparse Routing)
# ============================================================
class HydraLoRA(nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        r=8,
        num_experts=8,
        alpha=32,
        dropout=0.05,
        top_k=2
    ):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.scaling = alpha / r

        self.lora_A = nn.ModuleList([
            nn.Linear(in_features, r, bias=False)
            for _ in range(num_experts)
        ])
        self.lora_B = nn.ModuleList([
            nn.Linear(r, out_features, bias=False)
            for _ in range(num_experts)
        ])

        self.router = nn.Linear(in_features, num_experts)
        self.dropout = nn.Dropout(dropout)

        for A, B in zip(self.lora_A, self.lora_B):
            nn.init.kaiming_uniform_(A.weight, a=5 ** 0.5)
            nn.init.zeros_(B.weight)

        nn.init.zeros_(self.router.weight)
        nn.init.zeros_(self.router.bias)

    def forward(self, x):
        # Router logits (stable)
        logits = self.router(x)
        logits = logits - logits.max(dim=-1, keepdim=True).values
        logits = torch.clamp(logits, -20, 20)

        # Top-2 routing
        topk_vals, topk_idx = torch.topk(logits, k=self.top_k, dim=-1)
        gates = F.softmax(topk_vals, dim=-1)

        # Expert outputs
        expert_outs = []
        for A, B in zip(self.lora_A, self.lora_B):
            expert_outs.append(
                B(self.dropout(A(x))) * self.scaling
            )
        expert_outs = torch.stack(expert_outs, dim=-2)

        # Sparse combine
        out = torch.zeros_like(expert_outs[..., 0, :])
        for i in range(self.top_k):
            idx = topk_idx[..., i]
            gate = gates[..., i].unsqueeze(-1)
            selected = torch.gather(
                expert_outs,
                dim=-2,
                index=idx.unsqueeze(-1).unsqueeze(-1)
                    .expand(-1, -1, 1, expert_outs.size(-1))
            ).squeeze(-2)
            out += gate * selected

        return out

class HydraLoRALinear(nn.Module):
    def __init__(self, base_layer):
        super().__init__()
        self.base = base_layer
        self.hydra = HydraLoRA(
            base_layer.in_features,
            base_layer.out_features
        ).to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype
        )

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.base(x) + self.hydra(x)

# -------------------------
# Inject into Q, V projections
# -------------------------
def apply_hydralora(model):
    for name, module in model.named_modules():
        if name.endswith("q_proj") or name.endswith("v_proj"):
            parent = model
            *path, attr = name.split(".")
            for p in path:
                parent = getattr(parent, p)
            setattr(parent, attr, HydraLoRALinear(getattr(parent, attr)))

apply_hydralora(model)

# -------------------------
# Freeze non-Hydra params
# -------------------------
for n, p in model.named_parameters():
    p.requires_grad = ("hydra" in n.lower())

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

# ============================================================
# Dataset
# ============================================================
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding=False
    )

tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"]
)
tokenized.set_format(type="torch")

def collate_fn(batch):
    return {
        "input_ids": torch.nn.utils.rnn.pad_sequence(
            [x["input_ids"] for x in batch],
            batch_first=True,
            padding_value=tokenizer.pad_token_id
        )
    }

train_loader = DataLoader(
    tokenized["train"],
    batch_size=1,
    shuffle=True,
    collate_fn=collate_fn
)

# -------------------------
# Optimizer
# -------------------------
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5
)

# ============================================================
# Training loop
# ============================================================
model.train()

GRAD_ACC = 8
MAX_STEPS = 500
step = 0
skipped = 0

for batch in train_loader:
    input_ids = batch["input_ids"].to(device)

    if input_ids.numel() == 0 or input_ids.shape[1] < 2:
        skipped += 1
        continue

    outputs = model(input_ids=input_ids, labels=input_ids)
    loss = outputs.loss

    if torch.isnan(loss) or torch.isinf(loss):
        optimizer.zero_grad(set_to_none=True)
        skipped += 1
        continue

    loss = loss / GRAD_ACC
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        filter(lambda p: p.requires_grad, model.parameters()),
        max_norm=1.0
    )

    if (step + 1) % GRAD_ACC == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    if step % 50 == 0:
        print(f"Step {step} | Loss: {loss.item() * GRAD_ACC:.4f}")

    step += 1
    if step >= MAX_STEPS:
        break

print(f"✅ Training finished | Skipped batches: {skipped}")

# ============================================================
# ENTROPY METRICS (CORRECT, PAPER-FAITHFUL)
# ============================================================

num_experts = 8
expert_counts = torch.zeros(num_experts, device="cuda")

with torch.no_grad():
    for _ in range(200):
        x = torch.randn(
            4, 32, model.config.hidden_size,
            device="cuda", dtype=torch.bfloat16
        )
        logits = model.model.layers[0].self_attn.q_proj.hydra.router(x)
        _, topk_idx = torch.topk(logits, k=2, dim=-1)

        for i in range(2):
            expert_counts.scatter_add_(
                0,
                topk_idx[..., i].reshape(-1),
                torch.ones_like(
                    topk_idx[..., i].reshape(-1),
                    dtype=torch.float
                )
            )

probs = expert_counts / expert_counts.sum()
expert_entropy = -(probs * torch.log(probs + 1e-8)).sum()

print("\n=== ROUTING DIAGNOSTICS ===")
print("Expert usage probs:", probs.tolist())
print(f"Expert-usage entropy: {expert_entropy.item():.4f}")
print(f"Max possible entropy: {math.log(num_experts):.4f}")

# Optional: top-2 routing entropy
top2_entropy_vals = []
with torch.no_grad():
    x = torch.randn(
        4, 32, model.config.hidden_size,
        device="cuda", dtype=torch.bfloat16
    )
    logits = model.model.layers[0].self_attn.q_proj.hydra.router(x)
    topk_vals, _ = torch.topk(logits, k=2, dim=-1)
    probs = torch.softmax(topk_vals, dim=-1)
    top2_entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=-1).mean()

print(f"Top-2 routing entropy (max log2≈0.693): {top2_entropy.item():.4f}")

/home/xingjia2@andrew.cmu.edu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/xingjia2@andrew.cmu.edu/.conda/envs/research-env/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/xingjia2@andrew.cmu.edu/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:15<00:00,  8.00s/it]
/home/xingjia2@andrew.cmu.edu/.local/lib/python3.10/

Trainable params: 29,360,640
Trainable %: 0.4038%
Step 0 | Loss: 5.5004
Step 50 | Loss: 2.3772
Step 100 | Loss: 4.5257
Step 150 | Loss: 4.4995
Step 200 | Loss: 2.5779
Step 250 | Loss: 2.5299
Step 300 | Loss: 2.2384
Step 350 | Loss: 3.0112
Step 400 | Loss: 2.9213
Step 450 | Loss: 2.2783
✅ Training finished | Skipped batches: 297

=== ROUTING DIAGNOSTICS ===
Expert usage probs: [0.21529297530651093, 0.08359374850988388, 0.0768750011920929, 0.12306640297174454, 0.053105469793081284, 0.06230468675494194, 0.1489843726158142, 0.23677735030651093]
Expert-usage entropy: 1.9467
Max possible entropy: 2.0794
Top-2 routing entropy (max log2≈0.693): 0.6914


In [1]:
# ============================================================
# HYDRALORA — LONG TRAINING + ENTROPY LOGGING
# Mistral-7B | A100 80GB | BF16 | Paper-faithful
# ============================================================

import os, math
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader

# -------------------------
# Setup
# -------------------------
device = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True

model_name = "mistralai/Mistral-7B-v0.1"

# -------------------------
# Tokenizer
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False
)
tokenizer.pad_token = tokenizer.eos_token

# -------------------------
# Model
# -------------------------
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_flash_attention_2 = True
model.config.use_cache = False

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# ============================================================
# HydraLoRA (Top-2 Routing)
# ============================================================
class HydraLoRA(nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        r=8,
        num_experts=8,
        alpha=32,
        dropout=0.05,
        top_k=2
    ):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.scaling = alpha / r

        self.lora_A = nn.ModuleList([
            nn.Linear(in_features, r, bias=False)
            for _ in range(num_experts)
        ])
        self.lora_B = nn.ModuleList([
            nn.Linear(r, out_features, bias=False)
            for _ in range(num_experts)
        ])

        self.router = nn.Linear(in_features, num_experts)
        self.dropout = nn.Dropout(dropout)

        for A, B in zip(self.lora_A, self.lora_B):
            nn.init.kaiming_uniform_(A.weight, a=5 ** 0.5)
            nn.init.zeros_(B.weight)

        nn.init.zeros_(self.router.weight)
        nn.init.zeros_(self.router.bias)

    def forward(self, x):
        logits = self.router(x)
        logits = logits - logits.max(dim=-1, keepdim=True).values
        logits = torch.clamp(logits, -20, 20)

        topk_vals, topk_idx = torch.topk(logits, k=self.top_k, dim=-1)
        gates = F.softmax(topk_vals, dim=-1)

        expert_outs = []
        for A, B in zip(self.lora_A, self.lora_B):
            expert_outs.append(
                B(self.dropout(A(x))) * self.scaling
            )
        expert_outs = torch.stack(expert_outs, dim=-2)

        out = torch.zeros_like(expert_outs[..., 0, :])
        for i in range(self.top_k):
            idx = topk_idx[..., i]
            gate = gates[..., i].unsqueeze(-1)
            selected = torch.gather(
                expert_outs,
                dim=-2,
                index=idx.unsqueeze(-1).unsqueeze(-1)
                    .expand(-1, -1, 1, expert_outs.size(-1))
            ).squeeze(-2)
            out += gate * selected

        return out

class HydraLoRALinear(nn.Module):
    def __init__(self, base_layer):
        super().__init__()
        self.base = base_layer
        self.hydra = HydraLoRA(
            base_layer.in_features,
            base_layer.out_features
        ).to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype
        )

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.base(x) + self.hydra(x)

# -------------------------
# Inject into Q,V
# -------------------------
def apply_hydralora(model):
    for name, module in model.named_modules():
        if name.endswith("q_proj") or name.endswith("v_proj"):
            parent = model
            *path, attr = name.split(".")
            for p in path:
                parent = getattr(parent, p)
            setattr(parent, attr, HydraLoRALinear(getattr(parent, attr)))

apply_hydralora(model)

# -------------------------
# Freeze non-Hydra params
# -------------------------
for n, p in model.named_parameters():
    p.requires_grad = ("hydra" in n.lower())

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

# ============================================================
# Dataset
# ============================================================
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding=False
    )

tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"]
)
tokenized.set_format(type="torch")

def collate_fn(batch):
    return {
        "input_ids": torch.nn.utils.rnn.pad_sequence(
            [x["input_ids"] for x in batch],
            batch_first=True,
            padding_value=tokenizer.pad_token_id
        )
    }

train_loader = DataLoader(
    tokenized["train"],
    batch_size=1,
    shuffle=True,
    collate_fn=collate_fn
)

# -------------------------
# Optimizer
# -------------------------
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5
)

# ============================================================
# Training (LONG)
# ============================================================
model.train()

GRAD_ACC = 8
MAX_STEPS = 3000
LOG_EVERY = 500
SAVE_EVERY = 1000

num_experts = 8
step = 0
skipped = 0

for batch in train_loader:
    input_ids = batch["input_ids"].to(device)

    if input_ids.numel() == 0 or input_ids.shape[1] < 2:
        skipped += 1
        continue

    outputs = model(input_ids=input_ids, labels=input_ids)
    loss = outputs.loss

    if torch.isnan(loss) or torch.isinf(loss):
        optimizer.zero_grad(set_to_none=True)
        skipped += 1
        continue

    loss = loss / GRAD_ACC
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        filter(lambda p: p.requires_grad, model.parameters()),
        max_norm=1.0
    )

    if (step + 1) % GRAD_ACC == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    # -------------------------
    # Logging
    # -------------------------
    if step % LOG_EVERY == 0:
        print(f"Step {step} | Loss: {loss.item() * GRAD_ACC:.4f}")

        # ---- Expert-usage entropy ----
        expert_counts = torch.zeros(num_experts, device="cuda")

        with torch.no_grad():
            x = torch.randn(
                4, 32, model.config.hidden_size,
                device="cuda", dtype=torch.bfloat16
            )
            logits = model.model.layers[0].self_attn.q_proj.hydra.router(x)
            _, topk_idx = torch.topk(logits, k=2, dim=-1)

            for i in range(2):
                expert_counts.scatter_add_(
                    0,
                    topk_idx[..., i].reshape(-1),
                    torch.ones_like(
                        topk_idx[..., i].reshape(-1),
                        dtype=torch.float
                    )
                )

        probs = expert_counts / expert_counts.sum()
        entropy = -(probs * torch.log(probs + 1e-8)).sum()

        print(f"  Expert-usage entropy: {entropy.item():.4f}")

    # -------------------------
    # Checkpoint
    # -------------------------
    if step > 0 and step % SAVE_EVERY == 0:
        torch.save(
            model.state_dict(),
            f"hydralora_step_{step}.pt"
        )
        print(f"💾 Saved checkpoint at step {step}")

    step += 1
    if step >= MAX_STEPS:
        break

print(f"\n✅ Training finished | Skipped batches: {skipped}")


/home/xingjia2@andrew.cmu.edu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/xingjia2@andrew.cmu.edu/.conda/envs/research-env/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/xingjia2@andrew.cmu.edu/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.39s/it]
/home/xingjia2@andrew.cmu.edu/.local/lib/python3.10/

Trainable params: 29,360,640
Trainable %: 0.4038%
Step 0 | Loss: 5.6741
  Expert-usage entropy: 0.6931
Step 500 | Loss: 2.6134
  Expert-usage entropy: 1.8873
Step 1000 | Loss: 2.0346
  Expert-usage entropy: 1.8978
💾 Saved checkpoint at step 1000
Step 1500 | Loss: 2.0252
  Expert-usage entropy: 1.9079
Step 2000 | Loss: 1.5657
  Expert-usage entropy: 1.8897
💾 Saved checkpoint at step 2000
Step 2500 | Loss: 1.6705
  Expert-usage entropy: 1.9544

✅ Training finished | Skipped batches: 1723


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_linear_schedule_with_warmup
)
from datasets import load_dataset

# =========================
# CONFIG
# =========================
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
MAX_LENGTH = 512
BATCH_SIZE = 1
GRAD_ACC = 8
MAX_STEPS = 1500
LR = 2e-4
NUM_EXPERTS = 8
LORA_RANK = 8
DEVICE = "cuda"

torch.backends.cuda.matmul.allow_tf32 = True

# =========================
# TOKENIZER (FAST OFF)
# =========================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)
tokenizer.pad_token = tokenizer.eos_token

# =========================
# DATASET (CORRECT SPLIT)
# =========================
dataset = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft[:50000]"
)

def preprocess(example):
    user, assistant = "", ""
    for m in example["messages"]:
        if m["role"] == "user":
            user += m["content"]
        elif m["role"] == "assistant":
            assistant += m["content"]

    text = f"<s>[INST] {user.strip()} [/INST] {assistant.strip()}</s>"

    tokens = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

    input_ids = tokens["input_ids"]
    labels = input_ids.copy()

    inst_tokens = tokenizer(
        "[/INST]",
        add_special_tokens=False
    )["input_ids"]

    inst_end = None
    for i in range(len(input_ids) - len(inst_tokens)):
        if input_ids[i:i + len(inst_tokens)] == inst_tokens:
            inst_end = i + len(inst_tokens)
            break

    if inst_end is not None:
        labels[:inst_end] = [-100] * inst_end

    return {
        "input_ids": input_ids,
        "labels": labels
    }

dataset = dataset.map(
    preprocess,
    remove_columns=dataset.column_names
)

def collate_fn(batch):
    return {
        "input_ids": torch.tensor(
            [x["input_ids"] for x in batch],
            dtype=torch.long
        ),
        "labels": torch.tensor(
            [x["labels"] for x in batch],
            dtype=torch.long
        )
    }

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

# =========================
# HYDRALORA MODULES
# =========================
class HydraLoRA(nn.Module):
    def __init__(self, in_dim, out_dim, r, num_experts):
        super().__init__()
        self.router = nn.Linear(in_dim, num_experts, bias=False)

        self.lora_A = nn.ModuleList([
            nn.Linear(in_dim, r, bias=False) for _ in range(num_experts)
        ])
        self.lora_B = nn.ModuleList([
            nn.Linear(r, out_dim, bias=False) for _ in range(num_experts)
        ])

        for A, B in zip(self.lora_A, self.lora_B):
            nn.init.kaiming_uniform_(A.weight, a=5 ** 0.5)
            nn.init.zeros_(B.weight)

    def forward(self, x):
        gates = F.softmax(self.router(x), dim=-1)
        out = 0
        for i, (A, B) in enumerate(zip(self.lora_A, self.lora_B)):
            out = out + gates[..., i:i+1] * B(A(x))
        return out

class HydraLoRALinear(nn.Module):
    def __init__(self, base_layer, r, num_experts):
        super().__init__()
        self.base = base_layer
        self.hydra = HydraLoRA(
            base_layer.in_features,
            base_layer.out_features,
            r,
            num_experts
        )

    def forward(self, x):
        return self.base(x) + self.hydra(x)

# =========================
# LOAD MODEL
# =========================
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map={"": DEVICE}
)

# Freeze backbone
for p in model.parameters():
    p.requires_grad = False

# Inject HydraLoRA
for layer in model.model.layers:
    layer.self_attn.q_proj = HydraLoRALinear(
        layer.self_attn.q_proj,
        LORA_RANK,
        NUM_EXPERTS
    ).to(DEVICE, dtype=torch.bfloat16)

    layer.self_attn.v_proj = HydraLoRALinear(
        layer.self_attn.v_proj,
        LORA_RANK,
        NUM_EXPERTS
    ).to(DEVICE, dtype=torch.bfloat16)

# =========================
# PARAM COUNT
# =========================
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

# =========================
# OPTIMIZER
# =========================
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=100,
    num_training_steps=MAX_STEPS
)

# =========================
# TRAIN LOOP (STOP @ 1500)
# =========================
model.train()
optimizer.zero_grad()
step = 0

for batch in loader:
    batch = {k: v.to(DEVICE) for k, v in batch.items()}

    outputs = model(**batch)
    loss = outputs.loss / GRAD_ACC
    loss.backward()

    if (step + 1) % GRAD_ACC == 0:
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    if step % 500 == 0:
        print(f"Step {step} | Loss: {loss.item() * GRAD_ACC:.4f}")

    step += 1
    if step >= MAX_STEPS:
        break

# =========================
# SAVE BEST CHECKPOINT
# =========================
torch.save(model.state_dict(), "hydralora_step1500.pt")
print("✅ Training stopped at step 1500 — checkpoint saved")


Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.59s/it]


Trainable params: 29,360,128
Trainable %: 0.4038%
Step 0 | Loss: 1.9110
Step 500 | Loss: 0.7705
Step 1000 | Loss: 0.9225
✅ Training stopped at step 1500 — checkpoint saved


## HydraLoRA vs Standard LoRA

| Aspect | Standard LoRA | HydraLoRA |
|--------|---------------|-----------|
| **Experts** | 1 adapter per layer | 8 experts, top-2 routing |
| **Capacity** | Fixed rank (e.g. r=8) | Multi-expert specialization |
| **Trainable params** | ~0.1–0.2% | ~0.4% (more expressive) |
| **Routing** | N/A | Learned router selects 2 experts per token |
| **Use case** | General fine-tuning | Domain adaptation, multi-task |

**Why HydraLoRA?** The router learns to specialize experts for different input patterns, potentially improving quality with similar parameter budget.

In [5]:
# ============================================================
# EVALUATION: Perplexity + Router Diagnostics
# Run this AFTER cell 9 (model with HydraLoRA is in memory)
# Or load from checkpoint if starting fresh
# ============================================================

import torch
import torch.nn.functional as F
import math
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
CHECKPOINT_PATH = "hydralora_step1500.pt"
MAX_LENGTH = 512
BATCH_SIZE = 2
NUM_EXPERTS = 8
DEVICE = "cuda"
EVAL_SAMPLES = 200  # For perplexity

# If model not in memory (e.g. restarted kernel), load from checkpoint
if 'model' not in dir() or not hasattr(model.model.layers[0].self_attn.q_proj, 'hydra'):
    print("Loading model and checkpoint...")
    import sys, os
    sys.path.insert(0, os.getcwd())
    try:
        from src.hydralora import apply_hydralora
    except ImportError:
        raise ImportError("Run cells 4-9 first, or ensure src/hydralora.py is in path")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map={"": DEVICE}
    )
    apply_hydralora(model, top_k=2)
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt, strict=False)
    print("Checkpoint loaded.")
else:
    print("Using model from previous cell (already has HydraLoRA).")

model.eval()

# =========================
# PERPLEXITY on validation set
# =========================
print("\n=== PERPLEXITY EVALUATION ===")
eval_data = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
eval_data = eval_data.filter(lambda x: len(x["text"].strip()) > 0)

def tokenize_eval(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding=False)

eval_tokenized = eval_data.map(tokenize_eval, batched=True, remove_columns=["text"])
eval_tokenized = eval_tokenized.select(range(min(EVAL_SAMPLES, len(eval_tokenized))))

def collate_eval(batch):
    ids = [torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]
    return {"input_ids": torch.nn.utils.rnn.pad_sequence(ids, batch_first=True, padding_value=tokenizer.pad_token_id)}

eval_loader = DataLoader(eval_tokenized, batch_size=BATCH_SIZE, collate_fn=collate_eval)

total_loss = 0.0
total_tokens = 0

with torch.no_grad():
    for batch in eval_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        if input_ids.shape[1] < 2:
            continue
        outputs = model(input_ids=input_ids, labels=input_ids)
        n = (input_ids != tokenizer.pad_token_id).sum().item()
        total_loss += outputs.loss.item() * n
        total_tokens += n

avg_loss = total_loss / max(total_tokens, 1)
perplexity = math.exp(min(avg_loss, 20))  # Cap to avoid overflow
print(f"Validation loss: {avg_loss:.4f}")
print(f"Perplexity: {perplexity:.2f}")

# =========================
# ROUTER DIAGNOSTICS
# =========================
print("\n=== ROUTER DIAGNOSTICS ===")
expert_counts = torch.zeros(NUM_EXPERTS, device=DEVICE)

with torch.no_grad():
    for _ in range(100):
        x = torch.randn(4, 32, model.config.hidden_size, device=DEVICE, dtype=torch.bfloat16)
        logits = model.model.layers[0].self_attn.q_proj.hydra.router(x)
        _, topk_idx = torch.topk(logits, k=2, dim=-1)
        for i in range(2):
            expert_counts.scatter_add_(0, topk_idx[..., i].reshape(-1), torch.ones_like(topk_idx[..., i].reshape(-1), dtype=torch.float))

probs = expert_counts / expert_counts.sum()
entropy = -(probs * torch.log(probs + 1e-8)).sum().item()
print("Expert usage probs:", [f"{p:.3f}" for p in probs.tolist()])
print(f"Expert-usage entropy: {entropy:.4f} (max: {math.log(NUM_EXPERTS):.4f})")

ModuleNotFoundError: No module named 'hydra_lora'